# Spread EMBIG Colombia — modelo AR(1) sobre ln(spread)

Flujo: **datos → transformación → identificación → estacionalidad → estimación → diagnóstico → pronóstico → ajuste por volatilidad → póster**.

- Cada etapa tiene sus **funciones** (arriba) y una **llamada** (abajo) que guarda lo que necesita el póster en el diccionario `R`.
- Desde la identificación, el modelo es un **AR(1) con constante sobre ln(spread)** (d = 0).
- Las figuras usan paleta clara para integrarse con el póster.

## 0. Configuración

In [ ]:
from pathlib import Path
import warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

warnings.filterwarnings("ignore")

# --- parámetros del modelo ---
ORDEN = (1, 0, 0)          # AR(1) sobre ln(spread)
TREND = "c"                # con constante: la media de ln(spread) es el nivel de largo plazo
DIA_CORTE = 1              # día del mes de mar/jun/sep/dic para el dato trimestral
H = 2                      # horizonte de pronóstico (trimestres)
INI_OOS = 80               # primera observación del ejercicio fuera de muestra
AUTORES = ""               # <-- editar

# --- rutas ---
BASE_DIR = next((r for r in (Path.cwd(), *Path.cwd().parents)
                 if (r / "base de datos" / "Datos_EMBI.xlsx").is_file()), None)
if BASE_DIR is None:
    raise FileNotFoundError("No se encontró base de datos/Datos_EMBI.xlsx en la carpeta del proyecto ni en sus carpetas superiores.")
ARCHIVO = BASE_DIR / "base de datos" / "Datos_EMBI.xlsx"
SALIDA = BASE_DIR / "code" / "resultados_embi"
SALIDA.mkdir(parents=True, exist_ok=True)

# --- estilo gráfico claro (figuras y póster) ---
plt.style.use("default")
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
                     "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
C_AZUL, C_VERDE, C_ROJO = "#1F4E78", "#2E7D32", "#C0392B"

R = {}                     # resultados de cada etapa; el póster solo lee este diccionario

## 1. Datos: serie diaria → trimestral

In [ ]:
def cargar_diaria(archivo, hoja="Diarias"):
    """Lee la hoja diaria y devuelve la serie spread indexada por fecha (omite filas inválidas)."""
    raw = pd.read_excel(archivo, sheet_name=hoja, header=1)
    col_fecha, col_spread = raw.columns[:2]
    fecha = pd.to_datetime(raw[col_fecha], errors="coerce")
    spread = pd.to_numeric(raw[col_spread], errors="coerce")
    validas = fecha.notna() & spread.notna()
    if not validas.any():
        raise ValueError("No se encontraron filas diarias con fecha y spread válidos.")
    if not validas.all():
        print(f"Filas omitidas por fecha o spread inválido/vacío: {(~validas).sum()}")
    return pd.Series(spread[validas].to_numpy(), index=pd.DatetimeIndex(fecha[validas]), name="spread").sort_index()

def muestrear_trimestral(diaria, dia_corte=1):
    """Último dato disponible hasta el día `dia_corte` de mar, jun, sep y dic."""
    fechas, valores = [], []
    for anio in range(diaria.index.min().year, diaria.index.max().year + 1):
        for mes in (3, 6, 9, 12):
            corte = pd.Timestamp(anio, mes, dia_corte)
            if diaria.index.min() <= corte <= diaria.index.max():
                fechas.append(corte); valores.append(diaria.loc[:corte].iloc[-1])
    return pd.Series(valores, index=pd.DatetimeIndex(fechas), name="spread")

diaria = cargar_diaria(ARCHIVO)
y = muestrear_trimestral(diaria, DIA_CORTE)       # spread trimestral (pbs)
fechas = y.index
ly = np.log(y)                                    # ln(spread): serie del modelo (d = 0)
dly = ly.diff().dropna()                          # Δln(spread): solo referencia
dy = y.diff().dropna()

R.update(n=len(y), ini=f"{fechas[0]:%m/%Y}", fin=f"{fechas[-1]:%m/%Y}", ultimo=float(y.iloc[-1]))
print(f"Diaria: {diaria.index.min():%d/%m/%Y} a {diaria.index.max():%d/%m/%Y}")
print(f"Trimestral (corte día {DIA_CORTE}): {len(y)} obs., {fechas[0]:%d/%m/%Y} a {fechas[-1]:%d/%m/%Y}")

## 2. Transformación y varianza

In [ ]:
def grafico_transformacion(y, ly, dly, ruta):
    fig, ax = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
    ax[0].plot(y.index, y, color=C_AZUL, lw=1.8)
    ax[0].set_title("ANTES: Spread EMBIG Colombia (pbs), trimestral"); ax[0].set_ylabel("pbs")
    ax[1].plot(ly.index, ly, color=C_VERDE, lw=1.8); ax[1].axhline(ly.mean(), color="gray", lw=.8, ls="--")
    ax[1].set_title("ln(spread): serie del AR(1) (línea = media)"); ax[1].set_ylabel("ln")
    ax[2].plot(dly.index, dly, color=C_ROJO, lw=1.5); ax[2].axhline(0, color="gray", lw=.8)
    ax[2].set_title("Referencia: Δln(spread)"); ax[2].set_ylabel("Δln")
    plt.tight_layout(); fig.savefig(ruta, dpi=150); plt.show()

def tabla_varianza(series):
    """Levene entre mitades y ARCH(1), ARCH(4) sobre residuos de un AR(1) auxiliar."""
    def arch_ar1(s, lags):
        s = np.asarray(s)
        return het_arch(sm.OLS(s[1:], sm.add_constant(s[:-1])).fit().resid, nlags=lags)[1]
    for nombre, s in series.items():
        s = s.reset_index(drop=True); h = len(s) // 2
        print(f"{nombre:8s} var1={s[:h].var():.4f} var2={s[h:].var():.4f} "
              f"Levene p={stats.levene(s[:h], s[h:]).pvalue:.4f} | "
              f"ARCH(1) p={arch_ar1(s,1):.4f} ARCH(4) p={arch_ar1(s,4):.4f}")

grafico_transformacion(y, ly, dly, SALIDA / "fig1_transformacion.png")
print("--- Varianza constante ---")
tabla_varianza({"Nivel": y, "ln": ly, "Δln": dly})

## 3. Etapa 1 — Identificación (de aquí en adelante: ln(spread))

In [ ]:
def prueba_adf(s, reg):
    r = adfuller(s, regression=reg, autolag="AIC")
    print(f"ADF  reg={reg:2s} est={r[0]:.3f} p={r[1]:.4f} rezagos={r[2]} crit5%={r[4]['5%']:.3f}")
    return r[1]

def prueba_kpss(s, reg):
    r = kpss(s, regression=reg, nlags=int(4 * (len(s) / 100) ** 0.25))
    print(f"KPSS reg={reg:2s} est={r[0]:.3f} p={r[1]:.4f} crit5%={r[3]['5%']}")
    return r[1]

def grafico_fac_facp(y, ly, dly, ruta):
    fig, ax = plt.subplots(3, 2, figsize=(12, 12))
    for fila, (s, nom) in enumerate([(y, "spread original"), (ly, "ln(spread)"), (dly, "Δln(spread)")]):
        plot_acf(s, lags=24, ax=ax[fila, 0], title=f"FAC – {nom}")
        plot_pacf(s, lags=24, ax=ax[fila, 1], method="ywm", title=f"FACP – {nom}")
    plt.tight_layout(); fig.savefig(ruta, dpi=150); plt.show()

print("=== ETAPA 1: IDENTIFICACIÓN ===")
print("ln(spread):");  adf_ly = prueba_adf(ly, "c"); kpss_ly = prueba_kpss(ly, "c")
prueba_adf(ly, "ct"); prueba_kpss(ly, "ct")
print("Δln(spread):"); adf_dly = prueba_adf(dly, "c"); kpss_dly = prueba_kpss(dly, "c")
grafico_fac_facp(y, ly, dly, SALIDA / "fig2_FAC_FACP.png")
R["ident"] = dict(adf_ly=adf_ly, kpss_ly=kpss_ly, adf_dly=adf_dly, kpss_dly=kpss_dly)

## 4. Etapa 1b — Estacionalidad trimestral (s = 4) sobre el AR(1)

In [ ]:
def test_estacionalidad(ly, n_perm=300, semilla=0):
    """LR de dummies trimestrales en el AR(1), Kruskal-Wallis y FAC estacional de los residuos, fuerza STL."""
    base = ARIMA(ly, order=(1, 0, 0), trend="c").fit()
    X = pd.get_dummies(ly.index.quarter, drop_first=True).astype(float); X.index = ly.index
    est = ARIMA(ly, exog=X, order=(1, 0, 0), trend="c").fit()
    lr = 2 * (est.llf - base.llf)
    assert lr >= 0, "LR negativo: el modelo con dummies no convergió"
    p_lr = stats.chi2.sf(lr, df=X.shape[1])
    res = pd.Series(base.resid, index=ly.index)
    p_kw = stats.kruskal(*[res[res.index.quarter == q] for q in (1, 2, 3, 4)]).pvalue
    a = acf(res, nlags=12)

    def fuerza(r):
        s = STL(r, period=4, robust=True).fit()
        return max(0, 1 - s.resid.var() / (s.seasonal + s.resid).var())
    fs = fuerza(res)
    rng = np.random.default_rng(semilla)               # calibración: residuos barajados no tienen estacionalidad
    sims = np.array([fuerza(pd.Series(rng.permutation(res.values), index=res.index)) for _ in range(n_perm)])
    return dict(lr=lr, p_lr=p_lr, p_kw=p_kw, acf=a[[4, 8, 12]], banda=1.96 / np.sqrt(len(res)),
                fs=fs, p_perm=float(np.mean(sims >= fs)), fs_tipica=float(np.median(sims)))

print("=== ETAPA 1b: ESTACIONALIDAD EN AR(1) ===")
E = test_estacionalidad(ly)
print(f"LR dummies: LR={E['lr']:.2f}, gl=3, p={E['p_lr']:.4f}")
print(f"Kruskal-Wallis por trimestre: p={E['p_kw']:.4f}")
print("FAC residuos, rezagos 4, 8, 12:", E["acf"].round(3), "| banda ±", round(E["banda"], 3))
print(f"Fuerza STL={E['fs']:.3f} | p permutación={E['p_perm']:.3f} | típica sin estacionalidad={E['fs_tipica']:.3f}")
E["hay"] = min(E["p_lr"], E["p_kw"]) < 0.05
R["estac"] = E

## 5. Etapa 2 — Estimación del AR(1) y comparación fuera de muestra

In [ ]:
def ajustar_ar1(ly):
    modelo = ARIMA(ly, order=ORDEN, trend=TREND).fit()
    phi, mu = modelo.params["ar.L1"], modelo.params["const"]
    ic = modelo.conf_int().loc["ar.L1"].to_numpy()
    info = dict(phi=phi, mu=mu, p_phi=modelo.pvalues["ar.L1"], ic_phi=ic, ic_incluye_1=bool(ic[1] >= 1),
                vida_media=np.log(.5) / np.log(phi), nivel_lp=float(np.exp(mu)),
                aic=modelo.aic, bic=modelo.bic)
    return modelo, info

def rmse_oos(ly, orden, trend, ini=INI_OOS):
    """RMSE de pronóstico a 1 paso con ventana creciente (en ln)."""
    e = [ly.iloc[t + 1] - ARIMA(ly.iloc[:t + 1], order=orden, trend=trend).fit().forecast(1).iloc[0]
         for t in range(ini, len(ly) - 1)]
    return float(np.sqrt(np.mean(np.square(e))))

def comparar_modelos(ly):
    cands = [("AR(1) con constante", (1, 0, 0), "c"), ("ARMA(1,1) con constante", (1, 0, 1), "c"),
             ("Caminata (0,1,0)", (0, 1, 0), "n"), ("ARIMA(0,1,2)", (0, 1, 2), "n")]
    return (pd.DataFrame([(n, rmse_oos(ly, o, t)) for n, o, t in cands], columns=["Modelo", "RMSE"])
            .sort_values("RMSE").reset_index(drop=True))

print("=== ETAPA 2: ESTIMACIÓN ===")
modelo, M = ajustar_ar1(ly)
print(modelo.summary().tables[1])
print(f"φ={M['phi']:.3f}  IC95%=[{M['ic_phi'][0]:.3f}, {M['ic_phi'][1]:.3f}]  (¿incluye 1? {'sí' if M['ic_incluye_1'] else 'no'})")
print(f"Vida media={M['vida_media']:.1f} trimestres | nivel de largo plazo={M['nivel_lp']:.0f} pbs | AIC={M['aic']:.2f} BIC={M['bic']:.2f}")
tabla = comparar_modelos(ly)
print(f"\nRMSE a 1 paso fuera de muestra (desde la obs. {INI_OOS}):")
print(tabla.round(4).to_string(index=False))
R["modelo"], R["tabla"] = M, tabla

## 6. Etapa 3 — Diagnóstico de residuos

In [ ]:
def diagnosticar(modelo, ruta_fig):
    res = np.asarray(modelo.resid)
    lb = acorr_ljungbox(res, lags=[4, 8, 12], model_df=ORDEN[0] + ORDEN[2])["lb_pvalue"].to_numpy()
    d = dict(lb_p=lb, arch_p=het_arch(res, nlags=4)[1], jb_p=stats.jarque_bera(res)[1],
             asim=stats.skew(res), curt=stats.kurtosis(res, fisher=False))
    fig, ax = plt.subplots(2, 1, figsize=(10, 7))
    plot_acf(res, lags=24, ax=ax[0], title="FAC – residuos AR(1)")
    plot_pacf(res, lags=24, ax=ax[1], method="ywm", title="FACP – residuos AR(1)")
    plt.tight_layout(); fig.savefig(ruta_fig, dpi=150); plt.show()
    return d

print("=== ETAPA 3: DIAGNÓSTICO ===")
D = diagnosticar(modelo, SALIDA / "fig3_residuos.png")
print("Ljung-Box p (4,8,12):", D["lb_p"].round(3).tolist())
print("ARCH-LM(4) p:", round(D["arch_p"], 3))
print(f"Jarque-Bera p: {D['jb_p']:.2e} | asimetría={D['asim']:.2f} curtosis={D['curt']:.2f}")
R["diag"] = D

## 7. Etapa 4 — Pronóstico (IC 95% y 90%, en pbs)

In [ ]:
def pronosticar(modelo, fechas, h=H):
    fc = modelo.get_forecast(h)
    fut = [fechas[-1] + pd.DateOffset(months=3 * (i + 1)) for i in range(h)]
    return dict(fc=fc, fut=fut, punto=np.exp(fc.predicted_mean.values),
                ic95=np.exp(fc.conf_int(alpha=0.05).values), ic90=np.exp(fc.conf_int(alpha=0.10).values))

def panel_pronostico(ax, y, P, a95, a90, color, titulo, ref95=None, nivel_lp=None):
    fut, punto = P["fut"], P["punto"]
    ax.plot(y.index[-24:], y[-24:], marker="o", ms=3, color=C_AZUL, label="Observado")
    ax.plot([y.index[-1], *fut], [y.iloc[-1], *punto], "--o", color=color, label="Pronóstico (mediana)")
    ax.fill_between(fut, a95[:, 0], a95[:, 1], color=color, alpha=.15, label="IC 95%")
    ax.fill_between(fut, a90[:, 0], a90[:, 1], color=color, alpha=.30, label="IC 90%")
    if ref95 is not None:
        ax.plot(fut, ref95[:, 0], ":", color="gray", lw=1.3, label="IC 95% original"); ax.plot(fut, ref95[:, 1], ":", color="gray", lw=1.3)
    if nivel_lp is not None:
        ax.axhline(nivel_lp, color="gray", ls=":", lw=1, label="Nivel de largo plazo")
    for f, v in zip(fut, punto):
        ax.annotate(f"{v:.1f}", xy=(f, v), xytext=(0, 4), textcoords="offset points", ha="center", va="bottom",
                    color=color, fontsize=7, fontweight="bold", zorder=10)
    ax.set_title(titulo, fontsize=10); ax.set_ylabel("pbs"); ax.legend(fontsize=8)

print("=== ETAPA 4: PRONÓSTICO ===")
P = pronosticar(modelo, fechas)
pron = pd.DataFrame({"Fecha": [f"{f:%d/%m/%Y}" for f in P["fut"]], "Mediana": P["punto"],
                     "IC95_inf": P["ic95"][:, 0], "IC95_sup": P["ic95"][:, 1],
                     "IC90_inf": P["ic90"][:, 0], "IC90_sup": P["ic90"][:, 1]})
print(pron.round(1).to_string(index=False))
print(f"Último dato ({fechas[-1]:%d/%m/%Y}): {y.iloc[-1]:.0f} pbs | nivel de largo plazo: {M['nivel_lp']:.0f} pbs")
print("Nota: la mediana es exp(media de ln), no la media en pbs.")
fig, ax = plt.subplots(figsize=(10, 5))
panel_pronostico(ax, y, P, P["ic95"], P["ic90"], C_ROJO, "Pronóstico Spread EMBIG Colombia – AR(1) sobre ln", nivel_lp=M["nivel_lp"])
plt.tight_layout(); fig.savefig(SALIDA / "fig4_pronostico.png", dpi=150); plt.show()
R["pron"] = P

## 8. Etapa 5 — Intervalos ajustados por volatilidad

Solo cambia el **ancho** de los IC: el modelo, los coeficientes y la mediana no se modifican. Se detecta el corte donde cambia la varianza de los residuos, se calcula σ de la época reciente y se reescala el error estándar del pronóstico por σ_reciente / σ_total. El rango ajustado **supone que no habrá una nueva crisis**: preséntalo junto al original.

In [ ]:
CORTE_VOL = None     # None = detectar automáticamente; o fijar, p. ej. pd.Timestamp(2009, 3, 1)
MIN_OBS = 25         # mínimo de residuos a cada lado del corte

def ajustar_volatilidad(modelo, P, fechas, corte=None, min_obs=MIN_OBS):
    res = pd.Series(np.asarray(modelo.resid), index=fechas)
    if corte is None:
        cand = [(res.index[k], stats.levene(res.iloc[:k], res.iloc[k:]).pvalue) for k in range(min_obs, len(res) - min_obs)]
        corte = min(cand, key=lambda t: t[1])[0]
    ant, rec = res[res.index < corte], res[res.index >= corte]
    sig_tot, sig_rec = np.sqrt(modelo.params["sigma2"]), rec.std()
    factor = sig_rec / sig_tot
    mu_f, se = np.asarray(P["fc"].predicted_mean), np.asarray(P["fc"].se_mean) * factor
    def ic(alpha):
        z = stats.norm.ppf(1 - alpha / 2)
        return np.exp(np.column_stack([mu_f - z * se, mu_f + z * se]))
    return dict(corte=corte, lev=stats.levene(ant, rec).pvalue, sig_ant=ant.std(), sig_rec=sig_rec,
                sig_tot=sig_tot, factor=factor, ic95=ic(0.05), ic90=ic(0.10),
                jb_rec=stats.jarque_bera(rec)[1], n_rec=len(rec))

print("=== ETAPA 5: AJUSTE DE VOLATILIDAD ===")
V = ajustar_volatilidad(modelo, P, fechas, CORTE_VOL)
print(f"Corte: {V['corte']:%d/%m/%Y} | σ antes={V['sig_ant']:.3f} σ después={V['sig_rec']:.3f} | Levene p={V['lev']:.4f}")
print(f"Época reciente: n={V['n_rec']} | Jarque-Bera p={V['jb_rec']:.3f}")
print(f"σ total={V['sig_tot']:.3f} | σ reciente={V['sig_rec']:.3f} | factor={V['factor']:.2f}")
comp = pd.DataFrame({"Fecha": pron["Fecha"], "Mediana": P["punto"],
                     "IC95_orig": [f"{a:.0f}-{b:.0f}" for a, b in P["ic95"]], "IC95_ajust": [f"{a:.0f}-{b:.0f}" for a, b in V["ic95"]],
                     "IC90_orig": [f"{a:.0f}-{b:.0f}" for a, b in P["ic90"]], "IC90_ajust": [f"{a:.0f}-{b:.0f}" for a, b in V["ic90"]]})
print(comp.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
panel_pronostico(ax, y, P, V["ic95"], V["ic90"], C_VERDE, f"Pronóstico AR(1) – intervalos ajustados (σ desde {V['corte']:%m/%Y})", ref95=P["ic95"], nivel_lp=M["nivel_lp"])
plt.tight_layout(); fig.savefig(SALIDA / "fig5_pronostico_ajustado.png", dpi=150); plt.show()
fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
panel_pronostico(axs[0], y, P, P["ic95"], P["ic90"], C_ROJO, "ORIGINAL: σ de toda la muestra")
panel_pronostico(axs[1], y, P, V["ic95"], V["ic90"], C_VERDE, f"AJUSTADO: σ desde {V['corte']:%m/%Y}")
plt.tight_layout(); fig.savefig(SALIDA / "fig6_comparacion_original_vs_ajustado.png", dpi=150); plt.show()
R["vol"] = V

## 9. Etapa 6 — Póster (PDF, paleta clara)

In [ ]:
# Paleta clara
BG, CARD, BORDE, PILL = "#F4F7FB", "#FFFFFF", "#D5DCEB", "#CFE3FF"
INK, MUTED, ACC, OK, KO, HEAD = "#1B2A4A", "#5A6785", "#D97706", "#1E9E5A", "#D64545", "#DCE8FA"

def textos_poster(R):
    """Frases del póster, construidas con las cifras y p-valores de R (cambian solas si cambian los datos)."""
    I, E, M, D, V = R["ident"], R["estac"], R["modelo"], R["diag"], R["vol"]
    raiz = ("se rechaza raíz unitaria" if I["adf_ly"] < 0.05 else
            "no se rechaza raíz unitaria; el test tiene poca potencia con φ cercano a 1")
    ident = (f"El spread en niveles tiene varianza cambiante, por lo que se usa ln. ADF sobre ln(spread): p = {I['adf_ly']:.3f} ({raiz}); "
             f"KPSS: p = {I['kpss_ly']:.3f}. La FACP de ln(spread) tiene un solo pico en el rezago 1 (patrón AR(1)), así que se modela "
             f"en niveles de ln (d = 0). φ = {M['phi']:.3f}, IC 95% [{M['ic_phi'][0]:.2f}, {M['ic_phi'][1]:.2f}] "
             f"{'incluye' if M['ic_incluye_1'] else 'no incluye'} el 1.")
    est = (f"Estacionalidad trimestral (s=4) sobre el AR(1): LR de dummies p = {E['p_lr']:.2f}, Kruskal-Wallis p = {E['p_kw']:.2f}. "
           + ("Hay evidencia de patrón estacional; se contrasta un SARIMA." if E["hay"] else
              "No hay evidencia de estacionalidad; no se justifica un SARIMA."))
    modelo = (f"Vida media de un choque: {M['vida_media']:.1f} trimestres. Nivel de largo plazo: {M['nivel_lp']:.0f} pbs "
              f"(promedio de toda la muestra, crisis incluidas).")
    sup = ("Los IC asumen errores normales; la normalidad se rechaza (colas pesadas por las crisis), por eso se complementan con un rango ajustado por volatilidad."
           if D["jb_p"] < 0.05 else "Los residuos son compatibles con errores normales.")
    pron = (f"Último dato: {R['ultimo']:.0f} pbs. Rango ajustado: σ desde {V['corte']:%m/%Y} (σ = {V['sig_rec']:.3f} vs {V['sig_tot']:.3f}, "
            f"Levene p = {V['lev']:.3f}); supone que no habrá una nueva crisis. La mediana revierte hacia el nivel de largo plazo.")
    return dict(ident=ident, est=est, modelo=modelo, sup=sup, pron=pron)

def armar_poster(R, carpeta, salida_pdf, salida_png=None):
    T = textos_poster(R)
    M, D, P, V, tabla = R["modelo"], R["diag"], R["pron"], R["vol"], R["tabla"]
    fig = plt.figure(figsize=(16.5, 23.4), facecolor=BG)
    bg = fig.add_axes([0, 0, 1, 1]); bg.set_xlim(0, 1); bg.set_ylim(0, 1); bg.axis("off")

    def card(x0, y0, w, h, titulo):
        bg.add_patch(FancyBboxPatch((x0, y0), w, h, boxstyle="round,pad=0,rounding_size=0.012", fc=CARD, ec=BORDE, lw=1.5))
        bg.text(x0 + 0.02, y0 + h - 0.006, titulo, color=INK, fontsize=22, fontweight="bold", va="center",
                bbox=dict(boxstyle="round,pad=0.5", fc=PILL, ec="none"))
    def texto(x, yv, s, w=60, size=15, color=INK, **kw):
        bg.text(x, yv, textwrap.fill(s, w), color=color, fontsize=size, va="top", ha="left", **kw)
    def imagen(rect, nombre):
        ax = fig.add_axes(rect); ax.imshow(plt.imread(carpeta / nombre)); ax.axis("off")

    # encabezado
    bg.add_patch(Rectangle((0, 0.915), 1, 0.085, fc=HEAD, ec="none"))
    bg.text(0.5, 0.968, "SPREAD EMBIG DE COLOMBIA", color=INK, fontsize=44, fontweight="bold", ha="center", va="center")
    bg.text(0.5, 0.936, f"Modelo AR(1) sobre ln(spread) | serie trimestral {R['ini']} – {R['fin']} ({R['n']} obs.)",
            color=ACC, fontsize=20, ha="center", va="center", fontweight="bold")
    bg.text(0.03, 0.985, AUTORES, color=MUTED, fontsize=13, ha="left", va="center")

    # 1. serie original y transformación
    card(0.02, 0.655, 0.96, 0.250, "Serie original e identificación")
    imagen([0.03, 0.695, 0.34, 0.175], "fig1_transformacion.png")
    imagen([0.39, 0.695, 0.58, 0.175], "fig2_FAC_FACP.png")
    texto(0.03, 0.688, T["ident"], w=135, size=14)

    # 2. estimación y estacionalidad
    card(0.02, 0.490, 0.96, 0.155, "Estimación y estacionalidad")
    filas = "\n".join(f"{'► ' if i == 0 else '  '}{r.Modelo:24s} RMSE {r.RMSE:.4f}" for i, r in tabla.iterrows())
    bg.text(0.03, 0.598, "Error de pronóstico a 1 paso (fuera de muestra):\n" + filas, color=INK, fontsize=14, family="monospace", va="top")
    texto(0.03, 0.520, f"AR(1): φ = {M['phi']:.3f} (p = {M['p_phi']:.3f}); AIC {M['aic']:.2f}, BIC {M['bic']:.2f}. {T['modelo']}", w=70, size=13)
    texto(0.55, 0.598, T["est"], w=55, size=15)

    # 3. verificación de supuestos
    card(0.02, 0.270, 0.96, 0.210, "Verificación de supuestos")
    tests = [("Sin autocorrelación", f"Ljung-Box (4,8,12): p = {', '.join(f'{v:.2f}' for v in D['lb_p'])}", D["lb_p"].min() > 0.05),
             ("Homocedasticidad", f"ARCH-LM(4): p = {D['arch_p']:.3f}", D["arch_p"] > 0.05),
             ("Normalidad", f"Jarque-Bera: p = {D['jb_p']:.1e}", D["jb_p"] > 0.05)]
    for i, (nom, det, ok) in enumerate(tests):
        yy = 0.430 - i * 0.038
        bg.text(0.03, yy, "✓" if ok else "✗", color=OK if ok else KO, fontsize=26, fontweight="bold", va="center")
        bg.text(0.055, yy + 0.008, nom, color=INK, fontsize=16, fontweight="bold", va="center")
        bg.text(0.055, yy - 0.010, det, color=MUTED, fontsize=13, va="center")
    texto(0.03, 0.322, T["sup"], w=62, size=13, color=MUTED)
    imagen([0.50, 0.285, 0.47, 0.175], "fig3_residuos.png")

    # 4. pronóstico
    card(0.02, 0.090, 0.96, 0.170, "Pronóstico (pbs)")
    bg.text(0.03, 0.212, f"{'Fecha':12s}{'Mediana':>9s}{'IC95 orig.':>16s}{'IC95 ajust.':>16s}", color=ACC, fontsize=14, family="monospace", va="top", fontweight="bold")
    for i, f in enumerate(P["fut"]):
        a, b, c, d = P["ic95"][i, 0], P["ic95"][i, 1], V["ic95"][i, 0], V["ic95"][i, 1]
        bg.text(0.03, 0.190 - i * 0.022, f"{f:%d/%m/%Y}  {P['punto'][i]:8.0f}  {f'[{a:.0f}, {b:.0f}]':>15s}  {f'[{c:.0f}, {d:.0f}]':>15s}",
                color=INK, fontsize=14, family="monospace", va="top")
    texto(0.03, 0.140, T["pron"], w=62, size=13, color=MUTED)
    imagen([0.50, 0.095, 0.47, 0.150], "fig5_pronostico_ajustado.png")

    # 5. resultados
    card(0.02, 0.012, 0.96, 0.068, "Resultados")
    bg.text(0.5, 0.030, rf"$\ln S_t-{M['mu']:.3f}={M['phi']:.3f}\,(\ln S_{{t-1}}-{M['mu']:.3f})+\varepsilon_t$",
            color=INK, fontsize=26, ha="center", va="center")
    bg.text(0.98, 0.003, "Datos: Banco de la República (PD04715XD)", color=MUTED, fontsize=10, ha="right")

    fig.savefig(salida_pdf, facecolor=BG)
    if salida_png: fig.savefig(salida_png, dpi=60, facecolor=BG)
    plt.show()

armar_poster(R, SALIDA, SALIDA / "poster_embig.pdf", SALIDA / "poster_embig_preview.png")